# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [83]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [84]:
# Update DATA_PATH to wherever the raw NTSB file lives on your machine
DATA_PATH = "data/AviationData.csv"

df = pd.read_csv(DATA_PATH, encoding="latin-1", low_memory=False)
df.columns = df.columns.str.strip()
print(df.shape)
display(df.head())

df.info()

(88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,NC6404,Stinson,108-3,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,Fatal(4),Destroyed,NaN,N5069P,Piper,PA24-180,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,Fatal(3),Destroyed,NaN,N5142R,Cessna,172M,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,N1168J,Rockwell,112,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,Fatal(1),Destroyed,NaN,N15NY,Cessna,501,No,NaN,NaN,NaN,NaN,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [85]:
# Percentage of missing values per column
(df.isna().mean().sort_values(ascending=False) * 100).round(1)

Schedule                  85.8
Air.carrier               81.3
FAR.Description           64.0
Aircraft.Category         63.7
Longitude                 61.3
Latitude                  61.3
Airport.Code              43.6
Airport.Name              40.7
Broad.phase.of.flight     30.6
Publication.Date          15.5
Total.Serious.Injuries    14.1
Total.Minor.Injuries      13.4
Total.Fatal.Injuries      12.8
Engine.Type                8.0
Report.Status              7.2
Purpose.of.flight          7.0
Number.of.Engines          6.8
Total.Uninjured            6.7
Weather.Condition          5.1
Aircraft.damage            3.6
Registration.Number        1.6
Injury.Severity            1.1
Country                    0.3
Amateur.Built              0.1
Model                      0.1
Make                       0.1
Location                   0.1
Accident.Number            0.0
Investigation.Type         0.0
Event.Id                   0.0
Event.Date                 0.0
dtype: float64

In [86]:
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

**Assumptions for filtering**
- Keep events from **1983 onwards** (40-year maximum lifetime for a make/model).
- Keep **professional builds only** (`Amateur.Built == "No"`). Rows where this is missing are dropped because we cannot confirm they are professional builds.
- Keep **airplanes only**. `Aircraft.Category` is often blank in older records, so where it is missing but the engine type is fixed-wing (reciprocating, turbo prop, turbo fan, turbo jet) we impute `Airplane`. Turbo shaft is excluded because that is mostly helicopters. This is an approximation.

In [87]:
# Dates -> datetime, keep 1983 onwards
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")
print("Unparseable dates:", df["Event.Date"].isna().sum())
df = df[df["Event.Date"].dt.year >= 1983].copy()

# Inspect the relevant columns before filtering
for col in ["Amateur.Built", "Aircraft.Category"]:
    print(df[col].value_counts(dropna=False).head(15), "\n")

# Professional builds only
df["Amateur.Built"] = df["Amateur.Built"].astype("string").str.strip().str.title()
df = df[df["Amateur.Built"] == "No"].copy()

# Impute missing Aircraft.Category for fixed-wing engine types, then keep airplanes
engine_tmp = df["Engine.Type"].astype("string").str.strip().str.title()
df["Aircraft.Category"] = df["Aircraft.Category"].astype("string").str.strip().str.title()
fixed_wing_engines = ["Reciprocating", "Turbo Prop", "Turbo Fan", "Turbo Jet"]
impute_mask = df["Aircraft.Category"].isna() & engine_tmp.isin(fixed_wing_engines)
print("Imputed as Airplane:", impute_mask.sum())
df.loc[impute_mask, "Aircraft.Category"] = "Airplane"

df = df[df["Aircraft.Category"] == "Airplane"].copy()
print("Rows after filtering:", df.shape[0])

Unparseable dates: 0
Amateur.Built
No     76960
Yes     8228
NaN      101
Name: count, dtype: int64 

Aircraft.Category
NaN                  56566
Airplane             24441
Helicopter            3151
Glider                 456
Balloon                201
Weight-Shift           161
Gyrocraft              158
Powered Parachute       91
Ultralight              29
Unknown                 13
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
Name: count, dtype: int64 

Imputed as Airplane: 46180
Rows after filtering: 67627


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

**Derived columns**
- `total_occupants` = fatal + serious + minor + uninjured. This includes crew, since passengers cannot be separated from crew in this data.
- Rows where all four counts are missing carry no occupant information, so they are dropped. Otherwise a missing count is treated as 0 people.
- Rows with 0 occupants are dropped (a rate cannot be computed).
- `severe_injury_rate` = (fatal + serious) / `total_occupants`, between 0 and 1.

In [88]:
inj_cols = ["Total.Fatal.Injuries", "Total.Serious.Injuries",
            "Total.Minor.Injuries", "Total.Uninjured"]

df = df.dropna(subset=inj_cols, how="all").copy()
df[inj_cols] = df[inj_cols].fillna(0)

df["total_occupants"] = df[inj_cols].sum(axis=1)
df = df[df["total_occupants"] > 0].copy()

df["severe_injuries"] = df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"]
df["severe_injury_rate"] = df["severe_injuries"] / df["total_occupants"]

df["severe_injury_rate"].describe()

count    66705.000000
mean         0.263928
std          0.425223
min          0.000000
25%          0.000000
50%          0.000000
75%          0.500000
max          1.000000
Name: severe_injury_rate, dtype: float64

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

**Damage cleaning**
- Standardise the text (strip, title case).
- "Unknown" is converted to missing and those rows dropped, since we cannot tell whether the aircraft was destroyed.
- `destroyed` = 1 if damage is "Destroyed", otherwise 0.

In [89]:
# The NTSB column is "Aircraft.damage"; this finds it however it is capitalised
dmg_col = [c for c in df.columns if c.lower() == "aircraft.damage"][0]

df[dmg_col] = df[dmg_col].astype("string").str.strip().str.title()
print(df[dmg_col].value_counts(dropna=False))

df[dmg_col] = df[dmg_col].replace({"Unknown": pd.NA, "Unk": pd.NA})
df = df.dropna(subset=[dmg_col]).copy()
df["destroyed"] = (df[dmg_col] == "Destroyed").astype(int)

print("\nOverall destroyed fraction:", round(df["destroyed"].mean(), 3))

Aircraft.damage
Substantial    49442
Destroyed      13401
Minor           1970
<NA>            1818
Unknown           74
Name: count, dtype: int64[pyarrow]

Overall destroyed fraction: 0.207


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

**Make cleaning tasks**
1. Drop missing makes.
2. Uppercase, remove punctuation, collapse whitespace.
3. Strip company suffixes (INC, CO, CORP, COMPANY, AIRCRAFT, THE ...) so "CESSNA AIRCRAFT COMPANY" and "CESSNA" merge.
4. Apply a manual map for remaining spelling variants of the same manufacturer, found by reviewing `value_counts()` (e.g. AIRBUS INDUSTRIE -> AIRBUS, GATES LEARJET -> LEARJET). Distinct historical makers such as DOUGLAS and MCDONNELL DOUGLAS are kept separate.
5. Keep makes with at least 50 records.

In [90]:
df = df.dropna(subset=["Make"]).copy()

def clean_make(s):
    s = re.sub(r"[^\w\s]", " ", s.upper())
    s = re.sub(r"\b(THE|INC|CO|CORP|CORPORATION|COMPANY|LTD|LLC|AIRCRAFT|AIRPLANE|AIRPLANES)\b", " ", s)
    return re.sub(r"\s+", " ", s).strip()

# Variants of the same manufacturer found by reviewing value_counts() (extend if you spot more)
make_map = {"BEECHCRAFT": "BEECH", "DEHAVILLAND": "DE HAVILLAND",
            "AIRBUS INDUSTRIE": "AIRBUS", "DIAMOND IND": "DIAMOND",
            "GRUMMAN ACFT ENG COR SCHWEIZER": "GRUMMAN SCHWEIZER",
            "GRUMMAN AMERICAN AVN": "GRUMMAN AMERICAN", "CIRRUS": "CIRRUS DESIGN",
            "ERCOUPE ENG RESEARCH": "ERCOUPE", "GATES LEARJET": "LEARJET",
            "GULFSTREAM AMERICAN": "GULFSTREAM", "ROCKWELL INTERNATIONAL": "ROCKWELL"}

df["Make"] = df["Make"].astype(str).apply(clean_make).replace(make_map)
df = df[df["Make"] != ""].copy()
display(df["Make"].value_counts().head(25))

# Keep makes with at least 50 records
make_counts = df["Make"].value_counts()
df = df[df["Make"].isin(make_counts[make_counts >= 50].index)].copy()
print(df["Make"].nunique(), "makes kept;", df.shape[0], "rows")

Make
CESSNA               25077
PIPER                13784
BEECH                 4941
MOONEY                1295
BOEING                1181
GRUMMAN               1051
BELLANCA               967
AIR TRACTOR            882
AERONCA                605
ROBINSON               600
MAULE                  567
BELL                   542
CHAMPION               500
DE HAVILLAND           444
CIRRUS DESIGN          439
STINSON                421
ROCKWELL               396
AERO COMMANDER         395
LUSCOMBE               385
TAYLORCRAFT            365
NORTH AMERICAN         362
HUGHES                 318
SCHWEIZER              287
MCDONNELL DOUGLAS      282
AYRES                  269
Name: count, dtype: int64

63 makes kept; 60826 rows


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [91]:
df = df.dropna(subset=["Model"]).copy()

# Uppercase and remove spaces/hyphens/slashes so "737-200", "737 200" and "737200" collapse together
df["Model"] = (df["Model"].astype(str).str.upper().str.strip()
               .str.replace(r"[\s\-_/]+", "", regex=True))
df = df[df["Model"] != ""].copy()

# Are model labels unique to a make?
makes_per_model = df.groupby("Model")["Make"].nunique()
print("Model labels used by more than one make:", (makes_per_model > 1).sum(), "of", len(makes_per_model))
display(makes_per_model[makes_per_model > 1].sort_values(ascending=False).head(10))

# Unique plane-type identifier
df["aircraft_type"] = df["Make"] + " " + df["Model"]
df["aircraft_type"].value_counts().head(20)

Model labels used by more than one make: 222 of 3088


Model
G164A    6
500      5
G164B    5
7ECA     4
7KCAB    4
7AC      4
7GCB     4
AA1A     4
G164     4
AA1      4
Name: Make, dtype: int64

aircraft_type
CESSNA 152       2205
CESSNA 172       1602
CESSNA 172N      1078
PIPER PA28140     919
CESSNA 172M       778
CESSNA 150        738
CESSNA 172P       663
PIPER PA18        658
PIPER PA18150     615
CESSNA 182        599
PIPER PA28180     589
CESSNA 180        587
CESSNA 150M       550
PIPER PA28161     549
PIPER PA28181     534
BEECH A36         449
CESSNA 150L       437
PIPER PA38112     433
PIPER PA28        422
CESSNA 140        383
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

**Cleaning tasks**
- **Engine.Type**: standardise case; "Unknown"/"None" become missing.
- **Weather.Condition**: uppercase; "UNK"/"UNKNOWN" become missing, leaving VMC / IMC.
- **Number.of.Engines**: 0 is not valid for an airplane, so it becomes missing; stored as a nullable integer.
- **Purpose.of.flight** and **Broad.phase.of.flight**: strip, title case, "Unknown" becomes missing.

Remaining NaNs are left as they are; they are handled per analysis.

In [92]:
def clean_cat(series, unknowns):
    s = series.astype("string").str.strip().str.title()
    return s.replace({u: pd.NA for u in unknowns})

df["Engine.Type"] = clean_cat(df["Engine.Type"], ["Unknown", "Unk", "None"])
df["Purpose.of.flight"] = clean_cat(df["Purpose.of.flight"], ["Unknown", "Unk"])
df["Broad.phase.of.flight"] = clean_cat(df["Broad.phase.of.flight"], ["Unknown", "Unk"])

df["Weather.Condition"] = (df["Weather.Condition"].astype("string").str.strip().str.upper()
                           .replace({"UNK": pd.NA, "UNKNOWN": pd.NA}))

df["Number.of.Engines"] = df["Number.of.Engines"].replace(0, np.nan).astype("Int64")

for col in ["Engine.Type", "Weather.Condition", "Number.of.Engines",
            "Purpose.of.flight", "Broad.phase.of.flight"]:
    print(df[col].value_counts(dropna=False).head(10), "\n")

Engine.Type
Reciprocating      53880
Turbo Prop          2670
<NA>                2540
Turbo Fan           1332
Turbo Jet            381
Turbo Shaft           10
Geared Turbofan        1
Name: count, dtype: int64[pyarrow] 

Weather.Condition
VMC     53866
IMC      4719
<NA>     2229
Name: count, dtype: int64[pyarrow] 

Number.of.Engines
1       50685
2        8058
<NA>     1514
3         279
4         278
Name: count, dtype: Int64 

Purpose.of.flight
Personal               35213
Instructional           8566
<NA>                    6318
Aerial Application      3779
Business                3037
Positioning             1034
Other Work Use           602
Ferry                    567
Aerial Observation       461
Executive/Corporate      341
Name: count, dtype: int64[pyarrow] 

Broad.phase.of.flight
<NA>           15570
Landing        12317
Takeoff         9221
Cruise          7478
Maneuvering     5368
Approach        4713
Taxi            1546
Climb           1489
Descent         1314
Go-Arou

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [93]:
na_share = df.isna().mean().sort_values(ascending=False)
print(na_share.head(15).round(3))

# Drop columns that are more than 50% missing, except those needed for the analysis
protected = ["Engine.Type", "Weather.Condition", "Number.of.Engines", "Purpose.of.flight",
             "Broad.phase.of.flight", "severe_injury_rate", "destroyed"]
to_drop = [c for c in na_share[na_share > 0.5].index if c not in protected]
print("\nDropping:", to_drop)
df = df.drop(columns=to_drop)
df.shape

Schedule                 0.881
Air.carrier              0.836
FAR.Description          0.715
Longitude                0.635
Latitude                 0.635
Airport.Code             0.391
Airport.Name             0.362
Broad.phase.of.flight    0.256
Publication.Date         0.177
Purpose.of.flight        0.104
Report.Status            0.050
Engine.Type              0.042
Weather.Condition        0.037
Number.of.Engines        0.025
Registration.Number      0.003
dtype: float64

Dropping: ['Schedule', 'Air.carrier', 'FAR.Description', 'Longitude', 'Latitude']


(60814, 31)

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [94]:
df.to_csv("aviation_cleaned.csv", index=False)
print("Saved", df.shape)

Saved (60814, 31)
